In [11]:
"""
LLM Security Benchmark
This script measures how well an LLM classifies vulnerable and patched code pairs from CSV input.
"""

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import json
import os
import sys
import re
import time
import pandas as pd
from datetime import datetime
from pathlib import Path
from llmScan import *

In [12]:
# Create the benchmark-specific template and function
def create_benchmark_chain():
    """Create a chain specifically for benchmark boolean responses"""
    model, _ = get_model_and_chain()
    if model is None:
        return None, None
    
    benchmark_template = """<system>You are a security vulnerability analyzer for Python code.</system>

<user>
Your objective is to determine whether the provided Python code snippet is vulnerable or safe.
The snippet may be vulnerable or it may be safe. Classify the full snippet as a whole unit.

Examples of security vulnerabilities include:
- Unsafe use of untrusted input (e.g., from `request.args`, `request.form`, environment variables, external files)
- Dangerous function calls (e.g., `eval`, `exec`, `os.system`, `subprocess.run`, `pickle.loads`, `yaml.load`)
- Insecure file handling (e.g., `open` or `send_file` with user-controlled paths)
- Cryptographic mistakes (e.g., hardcoded keys, insecure algorithms)
- Web-specific issues (e.g., Cross-Site Scripting (XSS), CSRF vulnerabilities, Open Redirects)
- Hardcoded secrets (e.g., API keys, passwords, tokens)
- Misconfigurations (e.g., exposing debug mode, bad CORS policies)
- Omission of security features (e.g., missing input validation, hashing, secrets management)

Analyze this Python code snippet as a whole unit.
Do not score or classify line-by-line; return one final classification for the full snippet.

```python
{python_code}
```

Return your analysis as a single boolean value: True if vulnerable and False if safe
```

</user>
"""
    
    from langchain_core.prompts import ChatPromptTemplate
    prompt = ChatPromptTemplate.from_template(benchmark_template)
    chain = prompt | model
    
    model_name = getattr(model, 'model', 'unknown')
    
    return chain, model_name

benchmark_chain, model_name = create_benchmark_chain()
print(f"Benchmark chain created successfully with model: {model_name}" if benchmark_chain else "Failed to create benchmark chain")

Benchmark chain created successfully with model: gemma3:4b


In [14]:
# Define the benchmark test function with timing for code strings
def test_vulnerabilities_benchmark(python_code_text, include_raw_response=False):
    """
    Analyze a Python code string for security vulnerabilities with a boolean-style LLM response.

    Returns:
        tuple: (success, result, model_name, scan_time)
        tuple: (success, result, model_name, scan_time, raw_response) when include_raw_response=True
    """
    start_time = time.time()

    def _format_return(success, result, model, scan_time, raw_response=None):
        if include_raw_response:
            return success, result, model, scan_time, raw_response
        return success, result, model, scan_time

    try:
        if benchmark_chain is None:
            scan_time = time.time() - start_time
            return _format_return(False, None, "Error: No working LLM model available", scan_time, None)

        retries = 0
        response = None
        while retries < MAX_RETRIES:
            try:
                response = benchmark_chain.invoke({"python_code": str(python_code_text or "")})
                break
            except Exception as e:
                retries += 1
                print(f"LLM invocation failed (attempt {retries}/{MAX_RETRIES}): {e}")
                if retries >= MAX_RETRIES:
                    scan_time = time.time() - start_time
                    return _format_return(False, None, model_name, scan_time, None)
                print(f"Retrying in {RETRY_DELAY} seconds...")
                time.sleep(RETRY_DELAY)

        scan_time = time.time() - start_time

        if isinstance(response, bool):
            return _format_return(True, response, model_name, scan_time, response)
        if isinstance(response, str):
            response_lower = response.strip().lower()
            if response_lower == "true":
                return _format_return(True, True, model_name, scan_time, response)
            if response_lower == "false":
                return _format_return(True, False, model_name, scan_time, response)
            if "true" in response_lower and "false" not in response_lower:
                return _format_return(True, True, model_name, scan_time, response)
            if "false" in response_lower and "true" not in response_lower:
                return _format_return(True, False, model_name, scan_time, response)
            print(f"Ambiguous response: {response}")
            return _format_return(False, None, model_name, scan_time, response)

        print(f"Unexpected response type: {type(response)}")
        return _format_return(False, None, model_name, scan_time, response)

    except Exception:
        scan_time = time.time() - start_time
        return _format_return(False, None, model_name, scan_time, None)


def label_from_prediction(success, predicted_bool):
    if not success or predicted_bool is None:
        return "unknown"
    return "vulnerable" if predicted_bool else "safe"


print("test_vulnerabilities_benchmark function defined successfully!")

test_vulnerabilities_benchmark function defined successfully!


In [8]:
# Define CSV-driven benchmark runner
def run_benchmark_from_csv(
    csv_path,
    before_col="vulnerable_code",
    after_col="fixed_code",
    verbose=True,
):
    """
    Run LLM benchmark from a CSV where each row contains vulnerable and patched code.

    Adds classification, correctness, and completion columns for before/after snippets per row.
    """
    df = pd.read_csv(csv_path)

    required = [before_col, after_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}")

    output_df = df.copy()

    before_predictions = []
    after_predictions = []
    before_correct = []
    after_correct = []
    before_success = []
    after_success = []
    before_completed = []
    after_completed = []
    before_times = []
    after_times = []
    model_names = []

    for idx, row in output_df.iterrows():
        before_code = row.get(before_col, "")
        after_code = row.get(after_col, "")

        b_completed, b_pred, b_model, b_time = test_vulnerabilities_benchmark(before_code)
        a_completed, a_pred, a_model, a_time = test_vulnerabilities_benchmark(after_code)

        b_label = label_from_prediction(b_completed, b_pred)
        a_label = label_from_prediction(a_completed, a_pred)

        expected_before = "vulnerable"
        expected_after = "safe"
        b_is_correct = b_label == expected_before
        a_is_correct = a_label == expected_after

        before_predictions.append(b_label)
        after_predictions.append(a_label)
        before_correct.append(b_is_correct)
        after_correct.append(a_is_correct)
        before_success.append(b_is_correct)
        after_success.append(a_is_correct)
        before_completed.append(b_completed)
        after_completed.append(a_completed)
        before_times.append(round(b_time, 3))
        after_times.append(round(a_time, 3))
        model_names.append(a_model or b_model or "unknown")

        if verbose:
            print(
                f"Row {idx + 1}: before={b_label} (expected vulnerable), "
                f"after={a_label} (expected safe)"
            )

    output_df["expected_before"] = "vulnerable"
    output_df["expected_after"] = "safe"
    output_df["llm_model"] = model_names
    output_df["llm_classification_before"] = before_predictions
    output_df["llm_classification_after"] = after_predictions
    output_df["llm_correct_before"] = before_correct
    output_df["llm_correct_after"] = after_correct
    output_df["llm_success_before"] = before_success
    output_df["llm_success_after"] = after_success
    output_df["llm_completed_before"] = before_completed
    output_df["llm_completed_after"] = after_completed
    output_df["llm_scan_time_before_seconds"] = before_times
    output_df["llm_scan_time_after_seconds"] = after_times
    output_df["llm_pair_correct"] = output_df["llm_correct_before"] & output_df["llm_correct_after"]

    return output_df

print("run_benchmark_from_csv function defined successfully!")

run_benchmark_from_csv function defined successfully!


In [ ]:
# Run the CSV benchmark
print("Starting LLM CSV Security Benchmark...")

# Update this path to your benchmark input CSV
input_csv_path = "/Users/tyler/Projects/CVEfixes/sample.csv"

df_results = run_benchmark_from_csv(input_csv_path, verbose=False)

date_stamp = datetime.now().strftime("%Y%m%d")
raw_model = str(model_name) if model_name else "LLM"
model_tag = re.sub(r"[^A-Za-z0-9]+", "", raw_model) or "LLM"
output_file = f"benchmark{model_tag}{date_stamp}.csv"
df_results.to_csv(output_file, index=False)

print(f"Benchmark completed for {len(df_results)} rows.")
print(f"Results saved to: {output_file}")

In [ ]:
# Calculate and display pairwise metrics
print("=" * 60)
print("LLM CSV BENCHMARK RESULTS")
print("=" * 60)

total_rows = len(df_results)
before_acc = df_results["llm_correct_before"].mean() if total_rows else 0
after_acc = df_results["llm_correct_after"].mean() if total_rows else 0
pair_acc = df_results["llm_pair_correct"].mean() if total_rows else 0

print(f"Total row pairs analyzed: {total_rows}")
print(f"Before-code accuracy (expected vulnerable): {before_acc:.2%}")
print(f"After-code accuracy (expected safe): {after_acc:.2%}")
print(f"Pair accuracy (both correct in same row): {pair_acc:.2%}")

before_completed = df_results["llm_completed_before"].mean() if total_rows else 0
after_completed = df_results["llm_completed_after"].mean() if total_rows else 0
print(f"Before-code completion rate: {before_completed:.2%}")
print(f"After-code completion rate: {after_completed:.2%}")

before_unknown = (df_results["llm_classification_before"] == "unknown").sum() if total_rows else 0
after_unknown = (df_results["llm_classification_after"] == "unknown").sum() if total_rows else 0

print(f"Unknown classifications (before): {before_unknown}")
print(f"Unknown classifications (after): {after_unknown}")

avg_before_time = df_results["llm_scan_time_before_seconds"].mean() if total_rows else 0
avg_after_time = df_results["llm_scan_time_after_seconds"].mean() if total_rows else 0
print(f"Average scan time before-code: {avg_before_time:.3f}s")
print(f"Average scan time after-code: {avg_after_time:.3f}s")

if total_rows:
    models_used = sorted(df_results["llm_model"].dropna().unique().tolist())
    print(f"Model(s) in results: {', '.join(models_used)}")

In [ ]:
# Optional compact summary export
summary = {
    "total_row_pairs": len(df_results),
    "before_accuracy": float(df_results["llm_correct_before"].mean()) if len(df_results) else 0.0,
    "after_accuracy": float(df_results["llm_correct_after"].mean()) if len(df_results) else 0.0,
    "pair_accuracy": float(df_results["llm_pair_correct"].mean()) if len(df_results) else 0.0,
    "before_success_rate": float(df_results["llm_success_before"].mean()) if len(df_results) else 0.0,
    "after_success_rate": float(df_results["llm_success_after"].mean()) if len(df_results) else 0.0,
    "before_completed_rate": float(df_results["llm_completed_before"].mean()) if len(df_results) else 0.0,
    "after_completed_rate": float(df_results["llm_completed_after"].mean()) if len(df_results) else 0.0,
    "unknown_before": int((df_results["llm_classification_before"] == "unknown").sum()) if len(df_results) else 0,
    "unknown_after": int((df_results["llm_classification_after"] == "unknown").sum()) if len(df_results) else 0,
}

summary_df = pd.DataFrame([summary])
summary_file = "llm_csv_benchmark_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"Summary metrics saved to: {summary_file}")

In [ ]:
# Debug LLM Input/Output from CSV rows
def debug_llm_from_csv(
    csv_path,
    sample_pairs=1,
    start_row=0,
    random_sample=False,
    random_seed=42,
    before_col="vulnerable_code",
    after_col="fixed_code",
):
    df = pd.read_csv(csv_path)
    if df.empty:
        print("CSV is empty.")
        return

    total_rows = len(df)
    sample_pairs = int(sample_pairs) if sample_pairs is not None else 1
    if sample_pairs <= 0:
        print("sample_pairs must be >= 1")
        return

    if random_sample:
        sample_n = min(sample_pairs, total_rows)
        sampled_df = df.sample(n=sample_n, random_state=random_seed)
        selected_indices = sampled_df.index.tolist()
    else:
        if start_row < 0 or start_row >= total_rows:
            print(f"start_row {start_row} is out of range (0-{total_rows-1})")
            return
        end_row = min(start_row + sample_pairs, total_rows)
        selected_indices = list(range(start_row, end_row))

    print("=" * 60)
    print(f"LLM CSV DEBUG | Testing {len(selected_indices)} pair(s)")
    print("=" * 60)

    for i, row_index in enumerate(selected_indices, start=1):
        row = df.iloc[row_index]
        before_code = row.get(before_col, "")
        after_code = row.get(after_col, "")

        print("-" * 60)
        print(f"Sample {i}/{len(selected_indices)} | Row {row_index + 1}")

        b_completed, b_pred, b_model, b_time, b_raw = test_vulnerabilities_benchmark(before_code, include_raw_response=True)
        a_completed, a_pred, a_model, a_time, a_raw = test_vulnerabilities_benchmark(after_code, include_raw_response=True)
        b_label = label_from_prediction(b_completed, b_pred)
        a_label = label_from_prediction(a_completed, a_pred)
        b_success = b_label == "vulnerable"
        a_success = a_label == "safe"

        print(f"Before expected: vulnerable | predicted: {b_label} | success: {b_success} | completed: {b_completed} | model: {b_model} | time: {b_time:.3f}s")
        print(f"Before raw response: {repr(b_raw)}")
        print(f"After expected: safe       | predicted: {a_label} | success: {a_success} | completed: {a_completed} | model: {a_model} | time: {a_time:.3f}s")
        print(f"After raw response: {repr(a_raw)}")

# Example debug usage
input_csv_path = "sample2.csv"
# debug_llm_from_csv(input_csv_path, sample_pairs=3, start_row=0)
debug_llm_from_csv(input_csv_path, sample_pairs=5, random_sample=True, random_seed=42)

LLM CSV DEBUG | Testing 5 pair(s)
------------------------------------------------------------
Sample 1/5 | Row 1577
Before expected: vulnerable | predicted: vulnerable | success: True | completed: True | model: gemma3:4b | time: 22.428s
After expected: safe       | predicted: vulnerable | success: False | completed: True | model: gemma3:4b | time: 11.977s
------------------------------------------------------------
Sample 2/5 | Row 325
Before expected: vulnerable | predicted: vulnerable | success: True | completed: True | model: gemma3:4b | time: 12.087s
After expected: safe       | predicted: vulnerable | success: False | completed: True | model: gemma3:4b | time: 12.050s
------------------------------------------------------------
Sample 3/5 | Row 2124


KeyboardInterrupt: 